In [1]:
import os
import farsight as fs
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import synth_reader as sr
from astropy.io import fits
from astropy.table import Table
from joblib import Parallel, delayed

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from sklearn.mixture import GaussianMixture
from matplotlib.patches import Ellipse
from astropy.table import join

from sklearn.decomposition import PCA

import numpy as np
from matplotlib import pyplot as plt
import os
from astropy.table import Table
from astropy.cosmology import FlatwCDM
from getdist import plots, MCSamples
import zeus
from zeus import ChainManager
import numpy as np
import scipy.optimize as op
from multiprocessing import Pool




In [2]:
FADO = Table.read(fits.open('FadoGordonBest_Run.fits')[1])

class Data_get:
    def __init__(self,data,outliers,select):
        self.data = data
        self.outliers = outliers
        self.data_clean = None
        self.select = select #Debe ser una lista de Tuplas
        self.set = None
        self.remove_outliers()
        self.set_selection()


    def remove_outliers(self):
        tmp_dataFrame = self.data
        for x in self.outliers:
            tmp_dataFrame = tmp_dataFrame[tmp_dataFrame['TAB5_INDEX']!=x]
        self.data_clean = tmp_dataFrame 

    def set_selection(self):
        tmp_dataFrame = self.data_clean
        for s in range(len(self.select)):
            if self.select[s][1] == '==':
                tmp_dataFrame = tmp_dataFrame[tmp_dataFrame[self.select[s][0]]==self.select[s][2]]
            elif self.select[s][1] == '!=':
                tmp_dataFrame = tmp_dataFrame[tmp_dataFrame[self.select[s][0]]!=self.select[s][2]]
            elif self.select[s][1] == '>=':
                tmp_dataFrame = tmp_dataFrame[tmp_dataFrame[self.select[s][0]]>=self.select[s][2]]
            elif self.select[s][1] == '<=':
                tmp_dataFrame = tmp_dataFrame[tmp_dataFrame[self.select[s][0]]<=self.select[s][2]]
            elif self.select[s][1] == '>':
                tmp_dataFrame = tmp_dataFrame[tmp_dataFrame[self.select[s][0]]>self.select[s][2]]
            elif self.select[s][1] == '<':
                tmp_dataFrame = tmp_dataFrame[tmp_dataFrame[self.select[s][0]]<self.select[s][2]]
        self.set = tmp_dataFrame

    def figure_merit(self):
        x = np.array(self.set['ADEV'])
        y = np.array(self.set['chi2_red'])
        punto_ref = np.array([0, 1])
        distancias = np.sqrt((x - punto_ref[0])**2 + (y - punto_ref[1])**2)
        distancia_promedio = np.mean(distancias)

        return distancia_promedio


        
class Plotter(Data_get):
    def __init__(self,data,outliers,select):
        super().__init__(data,outliers,select)
        self.remove_outliers()
        self.set_selection()

DATA = Data_get(FADO,[56],
                [('RED_LAW','==','Gordon et al. 2003 - SMC Bar'),#]).set
                 ('SDSS_SNR','>=',10),
                 ('ADEV','>=',0),('ADEV','<=',25),
                 ('chi2_red','>=',0.5),('chi2_red','<=',1.5)]).set



DATA['lgSFR'] = np.log10(DATA['SFR'])
DATA['lgsSFR'] = np.log10(DATA['sSFR'])
for col in DATA.colnames:
    if np.issubdtype(DATA[col].dtype, np.number):
        mask = (DATA[col] == -999.0) | (DATA[col] == 999.0)
        DATA[col][mask] = np.random.normal(loc=0.1,scale=0.1)

tabla6_2014 = pd.read_csv('~/HIIGalaxies/Spectral_synthesis/Table6_art2014.csv')
def OH_to_Z(OH):
    return 10**(OH - 8.69)

tabla6_2014['Z_convOH'] = OH_to_Z(tabla6_2014['12+logO/H'])

TABLA6_astropy = Table.from_pandas(tabla6_2014)

tabla3_2014 = pd.read_csv('~/HIIGalaxies/Spectral_synthesis/Table3_art2014.csv')

TABLA3_astropy = Table.from_pandas(tabla3_2014)

merger_i = join(DATA, TABLA6_astropy, keys='TAB5_INDEX') # MERGER 1: Tabla 6 del articulo con los datos de FADO

merger_f = join(merger_i, TABLA3_astropy, keys='TAB5_INDEX') # MERGER 1: Tabla 3 del articulo con el merger anterior


# Calculos de proporciones de masas y demas.

burstAge_L = []
burstAge_M = []

Pop_5Myr  = []
Pop_10Myr = []
Pop_30Myr = []
Pop_100Myr = []




def nebular_fraction_at_lambda(wave, F_neb, F_star, lambda0=4861, window=20):
    mask = (wave > lambda0 - window) & (wave < lambda0 + window)

    neb = np.nanmedian(F_neb[mask])
    star = np.nanmedian(F_star[mask])

    return neb / (neb + star)

def nebular_fraction_integrated(wave, F_neb, F_star, lmin=3800, lmax=7000):
    mask = (wave >= lmin) & (wave <= lmax)

    neb_int = np.trapz(F_neb[mask], wave[mask])
    total_int = np.trapz((F_neb[mask] + F_star[mask]), wave[mask])

    return neb_int / total_int


fneb_4020_list = []
fneb_4861_list = []
fneb_opt_list = []



for x in range(len(merger_f)):

    obj = merger_f['TAB5_INDEX'][x]
    obj =str(obj)

    if len(obj)!=3:
        if len(obj) == 2:
            obj = '0' + obj
        else:
            obj = '00' + obj

    spec = fs.Nebulix(home = os.path.expanduser('~') +'/HIIGalaxies/FADO/output_4020/',
               file = f'{obj}GOR1.MILES150_1D.fits', 
               distance = merger_f[merger_f['TAB5_INDEX']>=int(obj)]['Z'][0])
    
    SSPs = spec.population_vector.copy()

    # Convertir masa corregida a unidades físicas
    SSPs['Mcor_jMo'] = (SSPs['Mcor_j'] / 100.0) * 10**spec.fado_ensemble['lg_Mp']

    # Máscara de población joven (t < 10 Myr)
    you_mask = SSPs['logage_j'] <= 7.0

    # Extraer columnas (astropy Table soporta esto directamente)
    x = SSPs['x_j'][you_mask]          # fracción de luz (%)
    m = SSPs['Mcor_jMo'][you_mask]     # masa
    logt = SSPs['logage_j'][you_mask]  # log edad

    # (Opcional) convertir % a fracción
    x = x / 100.0

    # Inicializar valores por defecto
    Io_by_L = np.nan
    Io_by_M = np.nan

    # Promedio ponderado por luz
    if np.sum(x) > 0:
        Io_by_L = np.sum(x * logt) / np.sum(x)

    # Promedio ponderado por masa
    if np.sum(m) > 0:
        Io_by_M = np.sum(m * logt) / np.sum(m)

    # Guardar resultados
    burstAge_L.append(Io_by_L)
    burstAge_M.append(Io_by_M)


    wave = spec.spectrum_bestfit['Lambda']
    F_star = spec.spectrum_bestfit['Flux_ste']
    F_total = spec.spectrum_bestfit['Flux_syn']

    F_neb = F_total - F_star
    F_neb = np.where(F_neb > 0, F_neb, 0)

    fneb_4020 = nebular_fraction_at_lambda(
        wave, F_neb, F_star, lambda0=4020, window=20
    )

    fneb_4861 = nebular_fraction_at_lambda(
        wave, F_neb, F_star, lambda0=4861, window=20
    )

    fneb_opt = nebular_fraction_integrated(
        wave, F_neb, F_star, lmin=3800, lmax=7000
    )

    fneb_4020_list.append(fneb_4020)
    fneb_4861_list.append(fneb_4861)
    fneb_opt_list.append(fneb_opt)

    x_5Myr  = np.sum(SSPs['x_j'][SSPs['logage_j'] <= np.log10(5e6)])
    x_10Myr = np.sum(SSPs['x_j'][SSPs['logage_j'] <= 7.0])
    x_30Myr = np.sum(SSPs['x_j'][SSPs['logage_j'] <= np.log10(3e7)])
    x_100Myr = np.sum(SSPs['x_j'][SSPs['logage_j'] <= 8.0])

    Pop_5Myr.append(x_5Myr)
    Pop_10Myr.append(x_10Myr)
    Pop_30Myr.append(x_30Myr)
    Pop_100Myr.append(x_100Myr)



merger_f['burstAge_L'] = np.array(burstAge_L)
merger_f['burstAge_M'] = np.array(burstAge_M)

merger_f['fneb_4020'] = np.array(fneb_4020_list)
merger_f['fneb_4861'] = np.array(fneb_4861_list)
merger_f['fneb_opt'] = np.array(fneb_opt_list)

merger_f['Pop_5Myr'] = np.array(Pop_5Myr)
merger_f['Pop_10Myr'] = np.array(Pop_10Myr)
merger_f['Pop_30Myr'] = np.array(Pop_30Myr)
merger_f['Pop_100Myr'] = np.array(Pop_100Myr)


merger_f['log_EWHb'] = np.log10(merger_f['EW(Hb)'])


In [3]:
cols = [
    'lgt_av_M',
    'Z_av_M',
    'lg_Mp',
    'logL(Hb)',
    'log_sigma(Hb)',
    'log_sigma([OIII])',
    'Z',
    'lg_QH',
    'log_EWHb','burstAge_L','burstAge_M','fneb_4020','fneb_opt','Pop_5Myr','Pop_10Myr','Pop_30Myr','Pop_100Myr','logR_u','12+logO/H','logSFR','lg_MppAGB','A_neb','T_e','n_e','A_v'
]

df = merger_f.to_pandas()   # o tu tabla ya mergeada

df = df[cols].apply(pd.to_numeric, errors='coerce')
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()

data =df

In [4]:
merger_f.columns

<TableColumns names=('SDSS_PMF','TAB5_INDEX','NED_NAME','PLATE','MJD','FIBERID','RA','DEC','Z','DESI_DR1','SDSS_SNR','converge','time','l_0','f_0','f_u','chi2_val','chi2_dev','chi2_red','L_dst','I_l','F_l','S_l','Cb_L','Cf_l','z','z_err','BPT_flag','BPT_Class','lgNII_Ha','elgNII_Ha','lgOIII_Hb','elgNII_Hb','T_e','n_e','A_v','eA_v','A_neb','eA_neb','v_0','ev_0','v_d','ev_d','t_av_L','et_av_L','t_av_M','et_av_M','lgt_av_L','elgt_av_L','lgt_av_M','elgt_av_M','Z_av_L','eZ_av_L','Z_av_M','etZ_av_M','lg_Me','elg_Me','lg_Mp','elg_Mp','lg_MepAGB','elg_MepAGB','lg_MppAGB','elg_MppAGB','tL_l0','etL_l0','tL_l0oneGyr','etL_l0oneGyr','tL_l0fivGyr','etL_l0fivGyr','lg_QH','elg_QH','lg_QHeI','elg_QHeI','lg_QHeII','elg_QHeII','pre_FHa','epre_FHa','pre_EWHa','epre_EWHa','pre_FHb','epre_FHb','pre_EWHb','epre_EWHb','obs_FHa','eobs_FHa','obs_EWHa','eobs_EWHa','obs_FHb','eobs_FHb','obs_EWHb','eobs_EWHb','tau_HaL','etau_HaL','tau_HaLext','etau_HaLext','tau_pAGBL','etau_pAGBL','tau_pAGBLext','etau_pAGBLext','

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Calculamos tus residuos observacionales reales usando tu alfa y beta óptimos:
alpha_best, beta_best = 33.2169717140405, 5.046006556006849 
log_L_obs = data['logL(Hb)']

# Tu variable objetivo (Target) es el residuo que quieres pulverizar:
data['residual'] = log_L_obs - (beta_best * data['log_sigma(Hb)'] + alpha_best)

# Definimos las matrices X (parámetros físicos) e y (residuos)
features_list = ['lgt_av_M', 'Z_av_M', 'lg_Mp','Z','lg_QH',
    'log_EWHb','burstAge_L','burstAge_M','fneb_4020','fneb_opt','Pop_5Myr','Pop_10Myr','Pop_30Myr','Pop_100Myr','logR_u','12+logO/H','logSFR','lg_MppAGB','A_neb','T_e','n_e','A_v']
X = data[features_list]
y = data['residual']

# 4. Inicializar y entrenar el Random Forest
# Usamos pocos estimadores y profundidad limitada por si tu muestra es pequeña (evitar overfitting)
rf = RandomForestRegressor(n_estimators=100, max_depth=4, random_state=42)
rf.fit(X, y)

# 5. Extraer e imprimir la importancia de los parámetros
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

print("Ranking de importancia de parámetros para explicar el scatter:")
for f in range(X.shape[1]):
    print(f"{f + 1}. {features_list[indices[f]]} ({importances[indices[f]]*100:.2f}%)")

# 6. Graficar los resultados
plt.figure(figsize=(10, 6))
plt.title("Importancia de los parámetros físicos en los residuos de L-$\sigma$")
plt.bar(range(X.shape[1]), importances[indices], align="center", color='teal')
plt.xticks(range(X.shape[1]), [features_list[i] for i in indices], rotation=45)
plt.ylabel("Importancia relativa")
plt.tight_layout()
plt.show()

Ranking de importancia de parámetros para explicar el scatter:
1. logR_u (23.42%)
2. 12+logO/H (10.55%)
3. n_e (10.54%)
4. A_neb (9.04%)
5. A_v (5.83%)
6. lgt_av_M (5.19%)
7. burstAge_L (4.93%)
8. Pop_10Myr (4.41%)
9. burstAge_M (3.29%)
10. fneb_4020 (3.21%)
11. T_e (2.54%)
12. Pop_30Myr (2.06%)
13. Pop_100Myr (2.01%)
14. log_EWHb (1.97%)
15. logSFR (1.88%)
16. Z (1.74%)
17. fneb_opt (1.71%)
18. lg_QH (1.53%)
19. lg_Mp (1.37%)
20. Z_av_M (1.19%)
21. lg_MppAGB (0.83%)
22. Pop_5Myr (0.77%)


In [7]:
import numpy as np
import pandas as pd
from pysr import PySRRegressor

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [8]:
data['OHratio'] = data['12+logO/H']

In [9]:
# Calculamos tus residuos observacionales reales usando tu alfa y beta óptimos:
alpha_best, beta_best = 33.2169717140405, 5.046006556006849 
log_L_obs = data['logL(Hb)']

# Tu variable objetivo (Target) es el residuo que quieres pulverizar:
data['residual'] = log_L_obs - (beta_best * data['log_sigma(Hb)'] + alpha_best)

# Definimos las matrices X (parámetros físicos) e y (residuos)
features = ['lgt_av_M','burstAge_L','burstAge_M','logR_u','OHratio','A_neb','n_e','A_v']
X = data[features].values
y = data['residual'].values

In [10]:
model = PySRRegressor(
    niterations=500,
    populations=30,
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["log10", "exp"], 
    
    # IMPORTANTE: 
    # Asegúrate de que los operadores en las restricciones (constraints) 
    # coincidan exactamente con los de la lista de arriba.
    constraints={
        'log10': 5, # Permite hasta 5 niveles de anidamiento dentro de un log
        'exp': 5,
        '/': (-1, 9)
    },
    
    # Si sigues teniendo problemas, quita 'nested_constraints' por un momento 
    # hasta que el modelo corra la primera vez.
    
    select_k_features=None,
    progress=True,
    loss="loss(prediction, target) = (prediction - target)^2"
)

print("Iniciando búsqueda genética de ecuaciones...")
model.fit(X, y, variable_names=features)

# ==========================================
# 3. VISUALIZACIÓN DE RESULTADOS
# ==========================================
# Imprime el set de ecuaciones candidatas ordenadas por complejidad y precisión (Score)
print("\n--- Ecuaciones encontradas por PySR ---")
print(model.equations_)

# Puedes exportar el frente de Pareto a un archivo de texto o LaTeX
model.equations_.to_csv("pysr_lsigma_residuals.csv")

# Para usar la mejor ecuación en tu código directamente:
best_equation_prediction = model.predict(X)

/home/holman/.conda/envs/cosmos/lib/python3.11/site-packages/pysr/sr.py:1036: FutureWarning: `loss` has been renamed to `elementwise_loss` in PySRRegressor. Please use that instead.
  warnings.warn(
/home/holman/.conda/envs/cosmos/lib/python3.11/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
Compiling Julia backend...


Iniciando búsqueda genética de ecuaciones...


[ Info: Started!



Expressions evaluated per second: 3.550e+05
Progress: 2139 / 15000 total iterations (14.260%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           1.460e-01  0.000e+00  y = -0.14276
3           1.412e-01  1.687e-02  y = n_e * -0.0011564
4           1.393e-01  1.347e-02  y = log10(logR_u + -2.0287)
5           1.354e-01  2.854e-02  y = -0.059633 / (logR_u - 2.2856)
6           1.350e-01  3.162e-03  y = log10(logR_u + -2.3852) * 0.28502
9           1.330e-01  4.914e-03  y = ((A_v / n_e) + -0.14984) / (logR_u - 1.9922)
10          1.319e-01  7.805e-03  y = log10(n_e) / ((-5.7295 - (logR_u / -0.39583)) / -0.088...
                                      491)
11          1.218e-01  7.971e-02  y = -0.15481 - (A_neb / (-15.051 - (n_e * (A_neb / -4.0434...
                                      )

[ Info: Final population:
[ Info: Results saved to:



--- Ecuaciones encontradas por PySR ---
    complexity      loss                                           equation  \
0            1  0.146039                                         -0.1427636   
1            3  0.141194                                n_e * -0.0011562482   
2            4  0.139304                         log10(logR_u + -2.0287359)   
3            5  0.135317                -0.054608922 / (logR_u - 2.2966523)   
4            6  0.134859             log10(logR_u + -2.3870678) * 0.2837228   
5            8  0.133598    (A_v + log10(logR_u + -2.3813515)) * 0.35760427   
6            9  0.130770  -0.2128785 - (A_v / (-0.3935537 - (n_e * 0.028...   
7           11  0.121740  -0.14752403 - (A_neb / (-15.046737 - ((A_neb *...   
8           13  0.104053  (-1.4022119 / (-14.92372 - ((n_e * (A_neb - 0....   
9           15  0.101262  ((-0.19150224 / (-15.064372 - (n_e * ((A_neb -...   
10          16  0.101074  (-1.5503812 / (-14.947482 - (n_e * ((A_neb - 0...   
11         

In [ ]:
import numpy as np
import pandas as pd
from pysr import PySRRegressor

# ==========================================
# 1. PREPARACIÓN DE DATOS (Mock para ejemplo)
# ==========================================
# Reemplaza esto con tu DataFrame real de STARLIGHT/FADO/CIGALE
np.random.seed(42)
n_samples = 100

data = pd.DataFrame({
    'log_sigma': np.random.uniform(1.2, 2.3, n_samples),
    'M_stellar': np.random.uniform(8.0, 11.0, n_samples),     # De CIGALE/STARLIGHT
    'Age_weighted': np.random.uniform(6.0, 9.0, n_samples),   # Edad log(t)
    'Z_gas': np.random.uniform(-1.5, 0.2, n_samples),         # Metalicidad
    'AV_dust': np.random.uniform(0.1, 1.5, n_samples),        # Atenuación
    'SFR_10Myr': np.random.uniform(-2.0, 1.5, n_samples)      # De FADO
})

# Calculamos tus residuos observacionales reales usando tu alfa y beta óptimos:
alpha_best, beta_best = 4.0, 40.5 
log_L_obs = alpha_best * data['log_sigma'] + beta_best + np.random.normal(0, 0.15, n_samples)

# Tu variable objetivo (Target) es el residuo que quieres pulverizar:
data['residual'] = log_L_obs - (alpha_best * data['log_sigma'] + beta_best)

# Definimos las matrices X (parámetros físicos) e y (residuos)
features = ['M_stellar', 'Age_weighted', 'Z_gas', 'AV_dust', 'SFR_10Myr']
X = data[features].values
y = data['residual'].values

# ==========================================
# 2. CONFIGURACIÓN Y EJECUCIÓN DE PYSR
# ==========================================
# Definimos los operadores de forma explícita
# Usamos 'np.log10' para que PySR lo reconozca como el operador logarítmico
model = PySRRegressor(
    niterations=100,
    populations=20,
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["log10", "exp"], 
    
    # IMPORTANTE: 
    # Asegúrate de que los operadores en las restricciones (constraints) 
    # coincidan exactamente con los de la lista de arriba.
    constraints={
        'log10': 5, # Permite hasta 5 niveles de anidamiento dentro de un log
        'exp': 5,
        '/': (-1, 9)
    },
    
    # Si sigues teniendo problemas, quita 'nested_constraints' por un momento 
    # hasta que el modelo corra la primera vez.
    
    select_k_features=None,
    progress=True,
    loss="loss(prediction, target) = (prediction - target)^2"
)

print("Iniciando búsqueda genética de ecuaciones...")
model.fit(X, y, variable_names=features)

# ==========================================
# 3. VISUALIZACIÓN DE RESULTADOS
# ==========================================
# Imprime el set de ecuaciones candidatas ordenadas por complejidad y precisión (Score)
print("\n--- Ecuaciones encontradas por PySR ---")
print(model.equations_)

# Puedes exportar el frente de Pareto a un archivo de texto o LaTeX
model.equations_.to_csv("pysr_lsigma_residuals.csv")

# Para usar la mejor ecuación en tu código directamente:
best_equation_prediction = model.predict(X)

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


/home/holman/miniconda3/envs/cosmos/lib/python3.11/site-packages/pysr/sr.py:1036: FutureWarning: `loss` has been renamed to `elementwise_loss` in PySRRegressor. Please use that instead.
  warnings.warn(


Iniciando búsqueda genética de ecuaciones...


/home/holman/miniconda3/envs/cosmos/lib/python3.11/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
Compiling Julia backend...
[ Info: Started!



Expressions evaluated per second: 1.040e+05
Progress: 628 / 2000 total iterations (31.400%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           2.170e-02  0.000e+00  y = -0.022726
3           2.082e-02  2.065e-02  y = log10(log10(M_stellar))
4           2.035e-02  2.264e-02  y = log10(M_stellar) + -0.9983
5           2.031e-02  2.128e-03  y = log10(log10(M_stellar)) / 0.48279
6           1.990e-02  2.057e-02  y = log10(log10(M_stellar)) / exp(Z_gas)
8           1.987e-02  7.123e-04  y = (log10(log10(M_stellar)) * 0.90102) / exp(Z_gas)
9           1.976e-02  5.395e-03  y = (log10(Age_weighted) * log10(log10(M_stellar))) / exp(...
                                      Z_gas)
10          1.957e-02  9.848e-03  y = (SFR_10Myr - Age_weighted) / exp((Z_gas + -2.9459) + M...
                 

[ Info: Final population:
[ Info: Results saved to:



--- Ecuaciones encontradas por PySR ---
    complexity      loss                                           equation  \
0            1  0.021698                                       -0.022726083   
1            3  0.020820                            log10(log10(M_stellar))   
2            4  0.020210                         -328.5068 / exp(M_stellar)   
3            6  0.019897               log10(log10(M_stellar)) / exp(Z_gas)   
4            7  0.019607  -0.18846336 / ((M_stellar + -7.9514575) * M_st...   
5            8  0.019557  (0.0072665783 / SFR_10Myr) - (150.27318 / exp(...   
6            9  0.019409  (-0.4813385 / (M_stellar * (M_stellar + -7.790...   
7           10  0.018738  (0.0071681193 / SFR_10Myr) - (130.61838 / exp(...   
8           12  0.018557  (0.0072665783 / SFR_10Myr) - ((150.27318 / exp...   
9           13  0.017988  (((1.0143971 - AV_dust) / (M_stellar / SFR_10M...   
10          15  0.017981  ((1.0192629 - AV_dust) / (M_stellar * (1.06459...   
11         

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe2 in position 4095: unexpected end of data

In [92]:
from astroquery.ipac.ned import Ned
import pandas as pd
import astropy.units as u
from astropy.cosmology import Planck18, z_at_value
from joblib import Parallel, delayed
import numpy as np

In [93]:
df  = pd.read_csv('/home/holman/PhysAstroNotes/AstroStatistics_with_R/Semana04/Dullo_MBH.dat', sep = '\s+')


df

,Galaxy,Type,mFUV,mFUVhi,mFUVlo,mNUV,mNUVhi,mNUVlo,m36,m36hi,...,DT36,DT36B,DT36D,DTFB,DTFV,DTNB,DTND,MBH,MBHlo,MBHhi
0,NGC0289,SBbc,12.84,0.39,-0.39,12.67,0.40,-0.40,10.55,0.38,...,0.92,0.14,0.05,NaN,0.66,NaN,0.66,7.38,-7.38,0.30
1,NGC0428,SABm,12.91,0.37,-0.37,12.65,0.37,-0.37,11.85,0.35,...,0.93,0.13,0.05,NaN,0.70,NaN,0.70,4.48,-4.48,0.37
2,NGC0613,SBbc,12.99,0.36,-0.36,12.46,0.36,-0.36,9.61,0.34,...,0.75,0.13,0.05,1.29,0.62,1.29,0.62,7.60,-0.35,0.35
3,NGC1042,SABc,13.08,0.39,-0.39,12.80,0.39,-0.39,11.10,0.37,...,0.97,0.13,0.05,NaN,0.70,NaN,0.70,4.40,-4.40,2.08
4,NGC1052,E4,16.81,0.24,-0.24,15.28,0.24,-0.24,10.05,0.23,...,NaN,0.01,NaN,0.23,NaN,0.23,NaN,8.24,-0.29,0.29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,NGC5846,E0-1,16.24,0.26,-0.26,15.18,0.25,-0.25,9.46,0.21,...,NaN,0.01,NaN,0.46,NaN,0.46,NaN,9.04,-0.06,0.06
63,NGC5879,SBbc,13.14,0.35,-0.35,12.79,0.35,-0.35,10.93,0.34,...,0.51,0.44,0.16,1.67,1.33,1.67,1.33,6.62,-6.62,0.28
64,NGC5921,SBbc,13.23,0.37,-0.37,12.76,0.37,-0.37,10.80,0.35,...,0.83,0.13,0.05,1.45,0.78,1.45,0.79,7.07,-7.07,0.42
65,NGC7418,SBc,13.35,0.36,-0.36,12.94,0.35,-0.35,10.82,0.33,...,0.97,0.13,0.04,NaN,0.61,NaN,0.61,5.18,-5.18,1.78


In [95]:
def redshift_convert(n,m):

    result_table = Ned.query_object(f"{df['Galaxy'].iloc[n]}")

    z = result_table['Redshift'][0]

    lum_dist = Planck18.luminosity_distance(z) / (1 * u.Mpc)


    #df.loc[n, "d_Mpc"] = float(lum_dist)

    #df.loc[n, "M3.6m"] = df['m36'].iloc[n] - 5 * np.log10(float(lum_dist)) - 25
    #{df['Galaxy'].iloc[n]}  
    print(f"{df['Galaxy'].iloc[n]} {float(lum_dist)}   {df['m36'].iloc[n] - 5 * np.log10(float(lum_dist)) - 25}")

    #return float(lum_dist)
    if m==1:
        return float(lum_dist)
    else:
        return df['m36'].iloc[n] - 5 * np.log10(float(lum_dist)) - 25

     # ,df['m36'].iloc[n] - 5 * np.log10(float(lum_dist)) - 25]

    #print(f"{float(lum_dist)} {df['m36'].iloc[n] - 5 * np.log10(float(lum_dist)) - 25}")

#dista = np.array(Parallel(n_jobs=-1)(delayed(redshift_convert)(i,1) for i in range(len(df))))
#M = np.array(Parallel(n_jobs=-1)(delayed(redshift_convert)(i,2) for i in range(len(df))))


for i in range(len(df)):
    redshift_convert(i,1)




NGC0289 24.177432957839756   -21.367050938490657
NGC0428 17.122486501707023   -19.317834162557883
NGC0613 21.97122657134599   -22.09927151316373
NGC1042 20.33327027526781   -20.441036167239368
NGC1052 23.15004676806433   -21.77275936361473
NGC1097 18.847847059650608   -22.486308745251907
NGC1300 23.418026110826162   -21.617751429492237
NGC1386 13.554610562350863   -20.150435221883633
NGC1493 15.603047227963213   -19.516047115186943
NGC2685 13.078331744345212   -19.612761747298894
NGC2748 21.89533383213805   -20.651757855736218
NGC2787 10.306771833442248   -20.21561330970814
NGC2903 8.137627558457632   -21.202489046283752
NGC2964 19.5436276593093   -20.735025899829015
NGC2974 28.022128375068476   -21.71750559165873
NGC3021 22.774916541661238   -20.16728397091181
NGC3031 -0.5759552956944844   nan
NGC3079 16.391631580168745   -21.81311092129895
NGC3115 10.151144232229209   -21.852574992396427
NGC3310 14.65435392561827   -20.139833381110765
NGC3368 13.15399703070554   -21.805288697442727
N

/tmp/ipykernel_6306/3303810907.py:14: RuntimeWarning: invalid value encountered in log10
  print(f"{df['Galaxy'].iloc[n]} {float(lum_dist)}   {df['m36'].iloc[n] - 5 * np.log10(float(lum_dist)) - 25}")
/tmp/ipykernel_6306/3303810907.py:14: RuntimeWarning: invalid value encountered in log10
  print(f"{df['Galaxy'].iloc[n]} {float(lum_dist)}   {df['m36'].iloc[n] - 5 * np.log10(float(lum_dist)) - 25}")


NGC4212 -1.3023812493669495   nan
NGC4245 12.873600193429917   -19.558500087082898
NGC4258 6.822703724927467   -21.27978256252379
NGC4278 9.608735780166894   -20.123331256857483
NGC4314 14.266947809332732   -20.43165536264937
NGC4321 23.31083145574541   -22.847788821528724
NGC4371 13.821716719143316   -20.26280993854399
NGC4374 15.068534361359722   -22.150355063919417
NGC4388 37.54349017573426   -22.762673218571642
NGC4472 14.53411895814848   -22.901943552320482
NGC4501 33.95540627456845   -23.8945446557172
NGC4548 7.1913555335899115   -19.464053801894323
NGC4593 37.06333431878505   -22.104722435290924
NGC4594 16.142113040470004   -23.8398019213696
NGC4596 28.098178477099225   -22.18339083386961
NGC4698 14.952736086036381   -20.793603340675173
NGC4736 4.554081686550809   -20.28200408143964
NGC4800 12.388532638925202   -19.525099346829137
NGC4826 6.050019048908984   -20.998783710306174
NGC5005 14.017610058165335   -21.563369873064808
NGC5018 41.91779704221665   -22.831992252804294
NGC50

In [86]:
M

array([-21.36705094, -19.31783416, -22.09927151, -20.44103617,
       -21.77275936, -22.48630875, -21.61775143, -20.15043522,
       -19.51604712, -19.61276175, -20.65175786, -20.21561331,
       -21.20248905, -20.7350259 , -21.71750559, -20.16728397,
                nan, -21.81311092, -21.85257499, -20.13983338,
       -21.8052887 , -21.20288509, -19.69258036, -19.95481575,
       -20.78526162, -21.7847273 , -20.34725657, -20.54805311,
       -20.10868965, -20.49582946, -20.81757775, -21.07380005,
                nan, -19.55850009, -21.27978256, -20.12333126,
       -20.43165536, -22.84778882, -20.26280994, -22.15035506,
       -22.76267322, -22.90194355, -23.89454466, -19.4640538 ,
       -22.10472244, -23.83980192, -22.18339083, -20.79360334,
       -20.28200408, -19.52509935, -20.99878371, -21.56336987,
       -22.83199225, -21.37619701, -21.95432179, -21.50990651,
       -19.78657595, -20.74337609, -21.96270403, -20.00006139,
       -21.3255898 , -22.07326301, -22.56547086, -19.35

In [87]:
df['d_Mpc'] = dista

df['M3.6m'] = M

df.to_csv('/home/holman/PhysAstroNotes/AstroStatistics_with_R/Semana04/Dist_incl.dat',sep = ' ',index=False)

df

,Galaxy,Type,mFUV,mFUVhi,mFUVlo,mNUV,mNUVhi,mNUVlo,m36,m36hi,...,DT36D,DTFB,DTFV,DTNB,DTND,MBH,MBHlo,MBHhi,d_Mpc,M3.6m
0,NGC0289,SBbc,12.84,0.39,-0.39,12.67,0.40,-0.40,10.55,0.38,...,0.05,NaN,0.66,NaN,0.66,7.38,-7.38,0.30,24.177433,-21.367051
1,NGC0428,SABm,12.91,0.37,-0.37,12.65,0.37,-0.37,11.85,0.35,...,0.05,NaN,0.70,NaN,0.70,4.48,-4.48,0.37,17.122487,-19.317834
2,NGC0613,SBbc,12.99,0.36,-0.36,12.46,0.36,-0.36,9.61,0.34,...,0.05,1.29,0.62,1.29,0.62,7.60,-0.35,0.35,21.971227,-22.099272
3,NGC1042,SABc,13.08,0.39,-0.39,12.80,0.39,-0.39,11.10,0.37,...,0.05,NaN,0.70,NaN,0.70,4.40,-4.40,2.08,20.333270,-20.441036
4,NGC1052,E4,16.81,0.24,-0.24,15.28,0.24,-0.24,10.05,0.23,...,NaN,0.23,NaN,0.23,NaN,8.24,-0.29,0.29,23.150047,-21.772759
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,NGC5846,E0-1,16.24,0.26,-0.26,15.18,0.25,-0.25,9.46,0.21,...,NaN,0.46,NaN,0.46,NaN,9.04,-0.06,0.06,25.415237,-22.565471
63,NGC5879,SBbc,13.14,0.35,-0.35,12.79,0.35,-0.35,10.93,0.34,...,0.16,1.67,1.33,1.67,1.33,6.62,-6.62,0.28,11.400846,-19.354685
64,NGC5921,SBbc,13.23,0.37,-0.37,12.76,0.37,-0.37,10.80,0.35,...,0.05,1.45,0.78,1.45,0.79,7.07,-7.07,0.42,21.957834,-20.907947
65,NGC7418,SBc,13.35,0.36,-0.36,12.94,0.35,-0.35,10.82,0.33,...,0.04,NaN,0.61,NaN,0.61,5.18,-5.18,1.78,21.511436,-20.843347
